# MindForge — Agent Playground

Manual scratchpad for playing with each agent as it lands. Not tests — just runnable cells.

**Setup:** copy `.env.example` → `.env`, fill in your Mistral key, then run Section 0 first.

---
**Sections**
- [0. Environment bootstrap](#0)
- [1. Cognee smoke test](#1)
- [2. Knowledge Curator Agent](#2)
- [3. Curriculum Architect Agent](#3)
- [4. Teacher Agent](#4)
- [5. Interviewer Agent](#5) ← add after Task 8.1
- [6. Orchestrator Agent](#6) ← add after Task 9.1
- [7. Integration: chained cross-agent flow](#7) ← add after Task 9

## 0. Environment bootstrap <a id='0'></a>

Run once per kernel session. Adds repo root to `sys.path`, loads `.env`, patches LiteLLM,
and configures Cognee for Mistral + local backends.

> **Why the LiteLLM patch?** Cognee passes internal kwargs (`dataset_name`, `dataset_id`)
> through to LiteLLM, which forwards them to the Mistral API — Mistral rejects them with
> a 422. The patch below strips those keys before every completion call.

In [3]:
import sys, os, pprint

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(REPO_ROOT, ".env"), override=True)

from mindforge.config import settings
print("LLM model     :", settings.llm_model)
print("Embedding     :", settings.embedding_model)
print("Mistral key   :", "SET" if settings.effective_mistral_api_key else "MISSING")

LLM model     : mistral/mistral-small-latest
Embedding     : mistral/mistral-embed
Mistral key   : SET


In [4]:
# ── LiteLLM patch ─────────────────────────────────────────────────────────────
# Strip Cognee-internal kwargs that the Mistral API rejects with HTTP 422.
import litellm

_STRIP_KEYS = {"dataset_name", "dataset_id"}

_orig_acompletion = litellm.acompletion
async def _clean_acompletion(*args, **kwargs):
    for k in _STRIP_KEYS:
        kwargs.pop(k, None)
    return await _orig_acompletion(*args, **kwargs)
litellm.acompletion = _clean_acompletion

_orig_completion = litellm.completion
def _clean_completion(*args, **kwargs):
    for k in _STRIP_KEYS:
        kwargs.pop(k, None)
    return _orig_completion(*args, **kwargs)
litellm.completion = _clean_completion

print("LiteLLM patched — dataset_name / dataset_id will be stripped from completions.")

LiteLLM patched — dataset_name / dataset_id will be stripped from completions.


In [ ]:
# ── Cognee configuration ───────────────────────────────────────────────────────
import cognee

# LLM — Mistral via LiteLLM 'custom' provider
await cognee.config.set_llm_config({
    "llm_provider": settings.llm_provider,
    "llm_model":    settings.llm_model,
    "llm_api_key":  settings.effective_mistral_api_key,
})

# Embeddings — Mistral mistral-embed
await cognee.config.set_embedding_config({
    "embedding_provider":   settings.embedding_provider,
    "embedding_model":      settings.embedding_model,
    "embedding_api_key":    settings.effective_mistral_api_key,
    "embedding_dimensions": settings.embedding_dimensions,
})

print("Cognee configured. LLM provider:", settings.llm_provider,
      "| model:", settings.llm_model)

TypeError: 'NoneType' object can't be awaited

## 1. Cognee smoke test <a id='1'></a>

Verify Cognee + Mistral + local backends all work before touching MindForge agents.
Exercises all 4 Cognee operations: `add` → `cognify` → `search` → `prune`.

In [7]:
# 1. Reset any previous state (safe on first run)
await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Step 1/5: Pruned old data")

✅ Step 1/5: Pruned old data


In [8]:
# 2. Add some text (remember)
smoke_text = (
    "The Perceptron was invented by Frank Rosenblatt in 1958. "
    "It is the simplest form of a neural network, consisting of "
    "a single layer of weights and a threshold activation function. "
    "Backpropagation was published by Rumelhart, Hinton, and Williams in 1986."
)
await cognee.add(smoke_text, dataset_name="smoke_test")
print("✅ Step 2/5: Added text to dataset")

✅ Step 2/5: Added text to dataset


In [9]:
# 3. Process (cognify = builds knowledge graph + embeddings)
await cognee.cognify(dataset_name="smoke_test")
print("✅ Step 3/5: Cognified (graph + embeddings built)")

principal_configuration table already exists, skipping creation
session_records table already exists, skipping creation
session_model_usage table already exists, skipping creation
✅ Step 3/5: Cognified (graph + embeddings built)


In [15]:
# 4. Search (recall)
from cognee.api.v1.search import SearchType

results = await cognee.search(
    query_type=SearchType.GRAPH_COMPLETION,
    query_text="Who invented the Perceptron?",
    datasets=["smoke_test"],
)
print("✅ Step 4/5: Search returned results:")
for r in results:
    print(f"   → {r}")

✅ Step 4/5: Search returned results:
   → {'dataset_id': UUID('22b16e53-2b85-5505-bc0f-d97924513297'), 'dataset_name': 'smoke_test', 'dataset_tenant_id': None, 'search_result': ['Frank Rosenblatt.']}


In [ ]:
# 5. Clean up (forget)
await cognee.prune.prune_data()
print("✅ Step 5/5: Cleaned up test data")
print("\n🎉 All Cognee operations work! Ready to test MindForge agents.")

## 2. Knowledge Curator Agent <a id='2'></a>

**What it does:** Ingests text/URLs, extracts concepts + prerequisite relationships via LLM,
persists to Cognee permanent memory.

| Direction | Cognee call | Scope |
|-----------|-------------|-------|
| Write | `cognee.remember()` | Permanent — dataset-scoped |
| Delete | `cognee.forget()` | Dataset or single item |

In [10]:
from mindforge.agents.knowledge_curator import KnowledgeCuratorAgent

curator = KnowledgeCuratorAgent()

SAMPLE_TEXT = """
Gradient descent is an optimisation algorithm that iteratively adjusts model parameters
in the direction of the negative gradient of the loss function. It is the foundation
of neural network training.

Backpropagation computes gradients of the loss with respect to every weight by applying
the chain rule backwards through the computational graph. It requires gradient descent
as a prerequisite.

The chain rule states that the derivative of a composed function equals the product of
the derivatives of its components. It directly underlies backpropagation.
"""

result = await curator.ingest_content(
    content=SAMPLE_TEXT,
    dataset="playground_dl",
    source_metadata={
        "title":  "Deep Learning Book",
        "author": "Goodfellow, Bengio, Courville",
        "year":   2016,
        "url":    "https://www.deeplearningbook.org",
    },
)

print("=== IngestionResult ===")
pprint.pprint(result)

=== IngestionResult ===
IngestionResult(concepts_count=3,
                relationships_count=2,
                concepts=['gradient_descent', 'backpropagation', 'chain_rule'],
                source_url='https://www.deeplearningbook.org',
                dataset='playground_dl')


In [14]:
# Ingest Wikipedia content via the REST API (avoids the 403 that the
# normal HTML page returns to non-browser user-agents).
# The /page/summary endpoint returns clean JSON — no scraping needed.
import httpx, json as _json

WIKI_API = "https://en.wikipedia.org/api/rest_v1/page/summary/Backpropagation"
async with httpx.AsyncClient(follow_redirects=True, timeout=20.0) as _c:
    _r = await _c.get(WIKI_API, headers={"Accept": "application/json"})
    _r.raise_for_status()
    _wiki = _r.json()

wiki_extract = _wiki.get("extract", "")
print(f"Fetched {len(wiki_extract)} chars from Wikipedia REST API")

url_result = await curator.ingest_content(
    content=wiki_extract,
    dataset="playground_dl",
    source_metadata={
        "title": _wiki.get("title", "Backpropagation"),
        "url":   _wiki.get("content_urls", {}).get("desktop", {}).get("page", WIKI_API),
    },
)
print("=== URL IngestionResult ===")
pprint.pprint(url_result)

HTTPStatusError: Client error '403 Forbidden' for url 'https://en.wikipedia.org/api/rest_v1/page/summary/Backpropagation'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

In [ ]:
# remove_item: delete one specific concept by ID.
await curator.remove_item("chain_rule")
print("chain_rule removed.")

In [ ]:
# remove_topic: wipe the entire dataset.
# Comment this out if you want data available for the Curriculum Architect section.
await curator.remove_topic("playground_dl")
print("Dataset playground_dl removed.")

## 3. Curriculum Architect Agent <a id='3'></a>

**What it does:** Queries the concept graph, topologically sorts by prerequisites,
returns a personalised learning path skipping mastered concepts.

| Direction | Cognee call | Scope |
|-----------|-------------|-------|
| Read | `cognee.recall()` | Permanent — concept graph + learner profile |

> **Note:** Re-ingest `SAMPLE_TEXT` above (comment out `remove_topic`) before running
> these cells so the concept graph has data to query.

In [ ]:
from mindforge.agents.curriculum_architect import CurriculumArchitectAgent

architect = CurriculumArchitectAgent()

path = await architect.generate_learning_path(
    goal="deep learning",
    learner_id="alice",
    dataset="playground_dl",
)

print(f"Learning path: {path.total_concepts} concepts, {path.estimated_hours:.0f}h estimated")
print()
for step in path.concepts:
    prereqs = ", ".join(step.prerequisites) if step.prerequisites else "none"
    print(f"  {step.order:2d}. [{step.difficulty:12s}] {step.concept_id}  (prereqs: {prereqs})")

In [ ]:
# Generate a path for a learner who has already mastered gradient_descent.
# The Curriculum Architect should skip that concept automatically.
from mindforge.models import LearnerProfile

# Simulate a stored profile by patching safe_recall to return a pre-built profile.
# In production this profile comes from Cognee permanent memory.
import mindforge.resilience as _res
_real_recall = _res.safe_recall

async def _mock_recall(query_text, **kwargs):
    if "learner profile" in query_text.lower():
        import dataclasses, json
        profile = LearnerProfile(
            learner_id="bob",
            mastered_concepts=["gradient_descent"],
            feedback_weights={"backpropagation": -2.0},  # bob is weak on backprop
        )
        return [dataclasses.asdict(profile)]
    return await _real_recall(query_text, **kwargs)

_res.safe_recall = _mock_recall

path_bob = await architect.generate_learning_path(
    goal="deep learning", learner_id="bob", dataset="playground_dl"
)

_res.safe_recall = _real_recall  # restore

print("Bob's path (gradient_descent skipped, backprop prioritised):")
for step in path_bob.concepts:
    print(f"  {step.order}. {step.concept_id}")

## 4. Teacher Agent <a id='4'></a>

**What it does:** Runs a Socratic dialogue — presents a small explanation chunk, asks a
probing question, evaluates the reply, and decides whether to advance to the next concept.

| Direction | Cognee call | Scope |
|-----------|-------------|-------|
| Read | `cognee.recall()` | Permanent — concept definition |
| Read | `cognee.recall()` | Session — last 5 dialogue turns |
| Write | `cognee.remember()` | Session — each Q&A turn |

In [ ]:
from mindforge.agents.teacher import TeacherAgent

teacher = TeacherAgent()
SESSION_ID = "playground_sess_001"
LEARNER_ID = "alice"

# teach_concept: retrieves concept from Cognee + session history,
# generates explanation + probing question via Mistral.
resp = await teacher.teach_concept(
    concept_id="backpropagation",
    learner_id=LEARNER_ID,
    session_id=SESSION_ID,
    dataset="playground_dl",
)

print("=== TeachingResponse ===")
print(f"Concept    : {resp.concept_id}")
print(f"Explanation: {resp.explanation}")
print(f"Question   : {resp.question}")
print(f"Source     : {resp.source}")
print(f"Turn       : {resp.turn}")

In [ ]:
# evaluate_response: LLM scores the learner's answer as poor / partial / good,
# persists the evaluation to Cognee session memory, returns advance flag.

# Try a good answer:
eval_good = await teacher.evaluate_response(
    learner_response="Gradients flow backwards from the output layer to the input layer using the chain rule.",
    expected_concept="backpropagation",
    session_id=SESSION_ID,
)
print("=== Good answer ===")
print(f"Level    : {eval_good.level}")
print(f"Score    : {eval_good.score:.2f}")
print(f"Feedback : {eval_good.feedback}")
print(f"Advance  : {eval_good.advance}")

In [ ]:
# Try a poor answer — advance should be False:
eval_poor = await teacher.evaluate_response(
    learner_response="I don't know, something about numbers?",
    expected_concept="backpropagation",
    session_id=SESSION_ID,
)
print("=== Poor answer ===")
print(f"Level    : {eval_poor.level}")
print(f"Score    : {eval_poor.score:.2f}")
print(f"Feedback : {eval_poor.feedback}")
print(f"Advance  : {eval_poor.advance}")

In [ ]:
# Simulate a 3-turn Socratic dialogue for one concept.
HARDCODED_TURNS = [
    ("gradient_descent",   "It moves parameters in the direction of the negative gradient."),
    ("backpropagation",    "It computes gradients layer by layer using the chain rule."),
    ("chain_rule",         "The derivative of a composite function is the product of derivatives."),
]

for concept_id, learner_answer in HARDCODED_TURNS:
    t = await teacher.teach_concept(
        concept_id=concept_id,
        learner_id=LEARNER_ID,
        session_id=SESSION_ID,
        dataset="playground_dl",
    )
    ev = await teacher.evaluate_response(
        learner_response=learner_answer,
        expected_concept=concept_id,
        session_id=SESSION_ID,
    )
    print(f"\n--- {concept_id} ---")
    print(f"  Q       : {t.question}")
    print(f"  Answer  : {learner_answer}")
    print(f"  Eval    : {ev.level} ({ev.score:.2f}) | advance={ev.advance}")
    print(f"  Feedback: {ev.feedback}")

## 5. Interviewer Agent <a id='5'></a>

> **Add cells here after Tasks 8.1–8.2 land.**

**What it does:** Runs an adaptive quiz — retrieves weak concepts, generates 5 questions,
adapts difficulty per answer, scores the session.

| Direction | Cognee call | Scope |
|-----------|-------------|-------|
| Read | `cognee.recall()` | Permanent — weak concept lookup |
| Write | `cognee.remember()` | Session — each answer + correctness |
| Read | `cognee.recall()` | Session — all answers for final score |

**Sample input:**
```python
iv = await interviewer.start_interview(
    learner_id="alice", session_id="sess_002",
    dataset="playground_dl", num_questions=5
)
```

**Sample output:**
```
InterviewSession(session_id='sess_002', total_questions=5, current_question=1,
                 question_text='What is the role of the learning rate in gradient descent?',
                 concept_id='gradient_descent', difficulty='medium')
```

## 6. Orchestrator Agent <a id='6'></a>

> **Add cells here after Tasks 9.1–9.2 land.**

**What it does:** Unified entry point — creates sessions, routes intent to the right agent,
calls `cognee.improve()` at session end to bridge session memory into the learner profile,
handles resets via `cognee.forget()`.

| Direction | Cognee call | Scope |
|-----------|-------------|-------|
| Read | `cognee.recall()` | Permanent — learner profile on session start |
| Write | `cognee.improve()` | Permanent — bridge session → learner profile |
| Delete | `cognee.forget()` | Learner profile reset or full wipe |

**Sample input:**
```python
session_id = await orchestrator.start_session(learner_id="alice")
resp = await orchestrator.route_request(
    intent="teach", learner_id="alice", session_id=session_id,
    payload={"concept_id": "backpropagation", "dataset": "playground_dl"}
)
await orchestrator.end_session(session_id, dataset="playground_dl")
```

## 7. Integration — chained cross-agent flow <a id='7'></a>

> **Activate when Orchestrator (Task 9) exists. Uncomment each step in order.**

Demonstrates how memory written by one agent becomes visible to the next:

1. **Ingest** → KnowledgeCurator writes to permanent memory (`remember`)
2. **Learn** → CurriculumArchitect reads concept graph (`recall`), returns ordered path
3. **Teach ×3** → TeacherAgent reads concepts + session history, stores each turn (`recall` + `remember`)
4. **End session** → Orchestrator calls `improve()`, session memory bridges into learner profile
5. **Interview** → InterviewerAgent reads updated weak concepts (`recall`), 5 questions
6. **Finish** → `InterviewResults` showing updated feedback weights
7. **Reset** → Orchestrator calls `forget()`, profile wiped

In [ ]:
# Step 0: imports — uncomment after all agents are implemented
# from mindforge.agents.knowledge_curator    import KnowledgeCuratorAgent
# from mindforge.agents.curriculum_architect import CurriculumArchitectAgent
# from mindforge.agents.teacher              import TeacherAgent
# from mindforge.agents.interviewer          import InterviewerAgent
# from mindforge.agents.orchestrator         import OrchestratorAgent
#
# LEARNER_ID  = "playground_user"
# DATASET     = "integration_dl"
#
# curator      = KnowledgeCuratorAgent()
# architect    = CurriculumArchitectAgent()
# teacher      = TeacherAgent()
# interviewer  = InterviewerAgent()
# orchestrator = OrchestratorAgent()
print("Uncomment imports once all agents are implemented.")

In [ ]:
# Step 1 — Ingest (cognee.remember)
# ingest_result = await curator.ingest_content(
#     content="Gradient descent ... Backpropagation ... Transformers ...",
#     dataset=DATASET,
#     source_metadata={"title": "Integration Test Source", "year": 2024},
# )
# print("Ingested:", ingest_result)
print("Step 1 placeholder.")

In [ ]:
# Step 2 — Generate learning path (cognee.recall)
# path = await architect.generate_learning_path(
#     goal="deep learning", learner_id=LEARNER_ID, dataset=DATASET
# )
# print("Path:", [c.concept_id for c in path.concepts])
print("Step 2 placeholder.")

In [ ]:
# Step 3 — Teach 3 turns (cognee.recall + cognee.remember)
# session_id = await orchestrator.start_session(learner_id=LEARNER_ID)
#
# HARDCODED_ANSWERS = [
#     "From output to input using the chain rule.",
#     "Weights are adjusted proportionally to their gradient.",
#     "The learning rate controls the step size.",
# ]
#
# for i, step in enumerate(path.concepts[:3]):
#     t = await teacher.teach_concept(
#         concept_id=step.concept_id, learner_id=LEARNER_ID,
#         session_id=session_id, dataset=DATASET,
#     )
#     print(f"\n--- Turn {i+1}: {step.concept_id} ---")
#     print("Explanation:", t.explanation)
#     print("Question   :", t.question)
#     ev = await teacher.evaluate_response(
#         learner_response=HARDCODED_ANSWERS[i],
#         expected_concept=step.concept_id,
#         session_id=session_id,
#     )
#     print("Eval       :", ev.level, "| advance:", ev.advance)
print("Step 3 placeholder.")

In [ ]:
# Step 4 — End session → cognee.improve() bridges session → learner profile
# await orchestrator.end_session(session_id, dataset=DATASET)
# print(f"Session {session_id} ended. improve() called.")
print("Step 4 placeholder.")

In [ ]:
# Steps 5 & 6 — Interview → finish (cognee.recall + cognee.remember)
# iv_session_id = await orchestrator.start_session(learner_id=LEARNER_ID)
# iv = await interviewer.start_interview(
#     learner_id=LEARNER_ID, session_id=iv_session_id,
#     dataset=DATASET, num_questions=5,
# )
# print("First question:", iv.question_text)
# results = await interviewer.finish_interview(session_id=iv_session_id, dataset=DATASET)
# print("Score:", results.score, "| Weak concepts:", results.weak_concepts)
# await orchestrator.end_session(iv_session_id, dataset=DATASET)
print("Steps 5–6 placeholder.")

In [ ]:
# Step 7 — Reset (cognee.forget)
# await orchestrator.reset_learner(LEARNER_ID)  # removes only this learner's profile
# await orchestrator.reset_all()                # nuclear — wipes everything
print("Step 7 placeholder.")